# TAPA memory & MFA benchmark (Colab)

Measures runtime and **peak RAM** (pipeline process + all subprocesses, i.e. what
Colab's ~12.7 GB limit actually sees) for one pipeline configuration, then lets
you compare runs.

**How to use**
1. Runtime → Change runtime type → **T4 GPU**.
2. Set the parameters in the next cell, then *Runtime → Run all*.
3. At the end, a `stats_*.json` (and `results_*.zip`) downloads automatically.
4. **Use a fresh runtime for each configuration** (Runtime → Disconnect and delete
   runtime), otherwise pip keeps the previously installed version.
5. After 2+ runs, use the *Compare runs* cell at the bottom (works in any runtime).

**Suggested run matrix**

| run | BRANCH | USE_MFA | MFA_SPLIT | what it shows |
|---|---|---|---|---|
| A | `master` | False | – | original pipeline baseline |
| B | `colab-memory-fixes` | False | – | memory fix alone |
| C | `colab-memory-fixes` | True | True | memory fix + per-segment MFA |
| D (optional) | `master` | True | – | original + whole-recording MFA — **expected to be slow, may hit TAPA's 30-min MFA timeout (falls back to CMUdict) or exhaust RAM; that's the failure mode being fixed** |


In [ ]:
BRANCH = "colab-memory-fixes"  # "master" = original code
USE_MFA = False                # install MFA (~5-10 min) and use forced alignment
MFA_SPLIT = True               # per-segment MFA (only exists on colab-memory-fixes)
DURATION_MIN = 20             # minutes of audio to benchmark (built by looping the sample)

In [ ]:
# Install TAPA from the selected branch (ffmpeg is preinstalled on Colab)
!pip install -q "git+https://github.com/sarmadchandio/tapa.git@{BRANCH}"
print("installed TAPA branch:", BRANCH)

In [ ]:
# Montreal Forced Aligner via Miniforge (only if USE_MFA)
if USE_MFA:
    import os
    if not os.path.exists("/opt/miniforge/bin/mfa"):
        !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh
        !bash /tmp/mf.sh -b -p /opt/miniforge > /dev/null
        !/opt/miniforge/bin/mamba install -y -q -c conda-forge montreal-forced-aligner > /dev/null
    !/opt/miniforge/bin/mfa model download acoustic english_us_arpa
    !/opt/miniforge/bin/mfa model download dictionary english_us_arpa
    !/opt/miniforge/bin/mfa version

In [ ]:
# Benchmark audio: download the sample once, then loop it with ffmpeg to the
# requested duration. Self-contained — no hosted files to go stale.
import os, subprocess
SAMPLE_URL = "https://www.youtube.com/watch?v=OQgE0ETV81s"
if not os.path.exists("bench_source.mp3"):
    !yt-dlp -q -x --audio-format mp3 -o "bench_source.%(ext)s" {SAMPLE_URL}
AUDIO = f"bench_{DURATION_MIN}min.mp3"
if not os.path.exists(AUDIO):
    # -stream_loop repeats the input; -t trims to the exact length wanted
    subprocess.run(["ffmpeg","-y","-v","error","-stream_loop","-1",
                    "-i","bench_source.mp3","-t",str(DURATION_MIN*60),
                    "-c","copy",AUDIO], check=True)
print(AUDIO, round(os.path.getsize(AUDIO)/1e6,1), "MB")

In [ ]:
# Run the pipeline once, sampling peak RSS of the whole process tree
import json, threading, time, psutil

def vmhwm_gb():
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmHWM"):
                return round(int(line.split()[1]) / 2**20, 2)
    return -1

class TreeSampler(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True)
        self.proc, self.peak, self.stop_flag = psutil.Process(), 0, False
    def run(self):
        while not self.stop_flag:
            total = 0
            try:
                total = self.proc.memory_info().rss
                for c in self.proc.children(recursive=True):
                    try:
                        total += c.memory_info().rss
                    except psutil.Error:
                        pass
            except psutil.Error:
                pass
            self.peak = max(self.peak, total)
            time.sleep(0.5)

from tapa.config import TAPAConfig
from tapa.pipeline import TAPAPipeline

has_split = "mfa_split_utterances" in TAPAConfig.__dataclass_fields__
kw = {"results_dir": "bench_results/"}
if USE_MFA:
    kw["mfa_bin"] = "/opt/miniforge/bin/mfa"
if has_split:
    kw["mfa_split_utterances"] = bool(USE_MFA and MFA_SPLIT)

pipe = TAPAPipeline(TAPAConfig(**kw))
sampler = TreeSampler(); sampler.start()

t0 = time.time(); pipe.load_models(); t_load = round(time.time() - t0, 1)
mem_models = vmhwm_gb()
t0 = time.time(); pipe.run(AUDIO); t_run = round(time.time() - t0, 1)
sampler.stop_flag = True

mfa_mode = ("split" if has_split and kw.get("mfa_split_utterances") else "single") if USE_MFA else "none"
TAG = f"{BRANCH}_{'mfa-' + mfa_mode if USE_MFA else 'nomfa'}"
stats = {
    "tag": TAG, "branch": BRANCH, "mfa": mfa_mode, "audio": AUDIO,
    "model_load_s": t_load, "run_s": t_run,
    "peak_rss_after_models_gb": mem_models,
    "peak_rss_after_run_gb": vmhwm_gb(),
    "peak_tree_rss_gb": round(sampler.peak / 2**30, 2),
    "colab_total_ram_gb": round(psutil.virtual_memory().total / 2**30, 1),
}
print(json.dumps(stats, indent=2))
with open(f"stats_{TAG}.json", "w") as f:
    json.dump(stats, f, indent=2)

In [ ]:
# Save + download this run's stats and result CSVs
!zip -qr results_{TAG}.zip bench_results stats_{TAG}.json
from google.colab import files
files.download(f"stats_{TAG}.json")
files.download(f"results_{TAG}.zip")

## Compare runs

Upload two or more `stats_*.json` (and optionally the matching `results_*.zip`)
from earlier runs. Prints a side-by-side table; with zips it also checks that
per-speaker phonetic averages agree between runs.

In [ ]:
from google.colab import files
import json, zipfile, os, csv, statistics

up = files.upload()
stats_list, zips = [], []
for name in up:
    if name.endswith(".json"):
        stats_list.append(json.loads(up[name]))
    elif name.endswith(".zip"):
        zips.append(name)

if stats_list:
    keys = ["tag", "run_s", "peak_tree_rss_gb", "peak_rss_after_run_gb", "model_load_s"]
    widths = [max(len(str(s.get(k, ""))) for s in stats_list + [dict(zip(keys, keys))]) for k in keys]
    print("  ".join(k.ljust(w) for k, w in zip(keys, widths)))
    for s in sorted(stats_list, key=lambda x: x.get("tag", "")):
        print("  ".join(str(s.get(k, "")).ljust(w) for k, w in zip(keys, widths)))

def load_avgs(d, fname, keycols):
    path = os.path.join(d, "bench_results", fname)
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return {tuple(r[k] for k in keycols): r for r in csv.DictReader(f)}

if len(zips) >= 2:
    dirs = []
    for z in zips:
        d = z[:-4] + "_x"
        zipfile.ZipFile(z).extractall(d)
        dirs.append(d)
    specs = [("vowel", ["speaker", "vowel"], ["mean_f1", "mean_f2"]),
             ("stop", ["speaker", "phone"], ["mean_vot_ms"]),
             ("fricative", ["speaker", "phone"], ["mean_cog"])]
    for a in range(len(dirs)):
        for b in range(a + 1, len(dirs)):
            print(f"\n{zips[a]} vs {zips[b]}:")
            for label, keycols, cols in specs:
                fname = next((f for f in os.listdir(os.path.join(dirs[a], "bench_results"))
                              if f.endswith(f"{label}_averages.csv")), None)
                if not fname:
                    continue
                o, n = load_avgs(dirs[a], fname, keycols), load_avgs(dirs[b], fname, keycols)
                if not o or not n:
                    continue
                diffs = []
                for k in set(o) & set(n):
                    if int(o[k].get("n_tokens", 0)) < 10 or int(n[k].get("n_tokens", 0)) < 10:
                        continue
                    for c in cols:
                        try:
                            x, y = float(o[k][c]), float(n[k][c])
                        except (ValueError, KeyError):
                            continue
                        if max(abs(x), abs(y)) > 0:
                            diffs.append(abs(x - y) / max(abs(x), abs(y)) * 100)
                if diffs:
                    print(f"  {label}: median {statistics.median(diffs):.2f}% / "
                          f"mean {statistics.mean(diffs):.2f}% / max {max(diffs):.2f}% "
                          f"({len(diffs)} cells with >=10 tokens)")